# MJX 08 — Playground PPO: Ball-in-Cup

### Lab Description

A second MuJoCo Playground control task: **`BallInCup`** (DM Control Suite). A planar actuator must swing a ball on a string up into a cup. It is a **sparse-reward, dynamic** task — noticeably harder than balancing a cartpole, because the agent gets little feedback until it actually lands the ball — yet it still learns reliably in a few minutes on the AMD GPU.

We reuse the same stable recipe as MJX 07: pure-JAX physics (`impl="jax"`), a modest `num_envs`, and the highest matmul precision.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Train PPO on a sparse-reward swing-up task
- See how a harder reward landscape changes the learning curve
- Render the trained Ball-in-Cup policy

### Import libraries and apply the compatibility shim

Same setup as MJX 07: the `device_put_replicated` shim and highest matmul precision for stable training on this ROCm stack.

In [ ]:
import os, functools
os.environ["MUJOCO_GL"] = "egl"

import jax
import jax.numpy as jp
if not hasattr(jax, "device_put_replicated"):
    def _dpr(x, devices=None):
        n = len(devices) if devices is not None else jax.local_device_count()
        return jax.tree_util.tree_map(
            lambda a: jax.device_put(jp.broadcast_to(jp.asarray(a)[None], (n,) + jp.asarray(a).shape)), x)
    jax.device_put_replicated = _dpr
jax.config.update("jax_default_matmul_precision", "highest")

import numpy as np
import matplotlib.pyplot as plt
import imageio
from mujoco_playground import registry, wrapper
from mujoco_playground.config import dm_control_suite_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from IPython.display import Video

print("JAX devices:", jax.devices())

### Load Ball-in-Cup and build the PPO config

We load the task with pure-JAX physics and start from Playground's tuned config, overriding the batch and eval counts for this GPU. (Ball-in-Cup needs a few more timesteps than the cartpole because its reward is sparse.)

In [ ]:
ENV_NAME = "BallInCup"
env = registry.load(ENV_NAME, config_overrides={"impl": "jax"})

cfg = dm_control_suite_params.brax_ppo_config(ENV_NAME)
ppo_params = cfg.to_dict()
net_cfg = ppo_params.pop("network_factory", None)
ppo_params.update(
    num_timesteps=4_000_000,
    num_envs=256,
    batch_size=256,
    num_minibatches=8,
    num_evals=10,
)
print({k: ppo_params[k] for k in ["num_timesteps", "num_envs", "batch_size", "episode_length"]})

### Train

Run PPO on the GPU. The eval reward stays low while the agent flails, then jumps once it discovers how to swing the ball into the cup — the signature of a sparse-reward task.

In [ ]:
progress = []
def progress_fn(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    progress.append((int(step), r))
    print(f"step {int(step):>9}  eval reward {r:8.1f}")

train_fn = functools.partial(ppo.train, **ppo_params)
if net_cfg:
    train_fn = functools.partial(train_fn,
        network_factory=functools.partial(ppo_networks.make_ppo_networks, **net_cfg))

make_inference_fn, params, _ = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    progress_fn=progress_fn,
    seed=0,
)
print("training done")

### Plot the learning curve

In [ ]:
steps, rewards = zip(*progress)
plt.figure(figsize=(7, 3))
plt.plot(steps, rewards, marker="o")
plt.xlabel("environment steps"); plt.ylabel("eval episode reward")
plt.title(f"PPO learning curve ({ENV_NAME})"); plt.grid(True); plt.show()

### Render the trained policy

We roll out the trained policy and render it swinging the ball into the cup.

In [ ]:
os.makedirs("output/videos", exist_ok=True)
inference = jax.jit(make_inference_fn(params))
reset, step = jax.jit(env.reset), jax.jit(env.step)

rng = jax.random.PRNGKey(1)
state = reset(rng)
trajectory = [state]
for _ in range(250):
    rng, k = jax.random.split(rng)
    action, _ = inference(state.obs, k)
    state = step(state, action)
    trajectory.append(state)

frames = np.asarray(env.render(trajectory, height=240, width=320))
out = "output/videos/mjx08_ball_in_cup.mp4"
imageio.mimsave(out, list(frames), fps=30)
print("saved", frames.shape[0], "frames ->", out)

### Watch the result

In [ ]:
Video(url="output/videos/mjx08_ball_in_cup.mp4")

## Conclusions

PPO solved a sparse-reward swing-up task with the same recipe that balanced the cartpole — only the timestep budget changed. MJX 09 takes the final step to a real robot arm: the Franka Panda pick-and-place, where stability tuning becomes essential.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT